# **Working/Testing with our deployed `research_graph`**

In [1]:
from langgraph_sdk import get_client

url = "http://localhost:8123"
client = get_client(url=url)


In [3]:
thread = await client.threads.create()
thread

{'thread_id': '01a09fd0-16e9-7b81-bb91-d0c179ea5880',
 'created_at': '2026-09-14T12:06:49.952509+00:00',
 'updated_at': '2026-09-14T12:06:49.952509+00:00',
 'state_updated_at': '2026-09-14T12:06:49.952509+00:00',
 'metadata': {},
 'config': {},
 'error': None,
 'status': 'idle',
 'values': None,
 'interrupts': {}}

In [4]:
runs = await client.runs.list(thread["thread_id"])
runs

[]

In [5]:
user_input = {
    "topic":"defence training for students in schools",
    "max_num_analyst":3
}

config = {"configurable":{"user_id":"shamoon"}}
graph_name = "research_graph"

async for chunk in client.runs.stream(thread["thread_id"],
                                      graph_name,
                                      input=user_input,
                                      stream_mode="messages-tuple",
                                      config=config):
    if chunk.event == "messages":
        print("".join(data_item['content'] for data_item in chunk.data if 'content' in data_item), end="", flush=True)

{"analysts":[{"name":"Dr. Priya Nanda","role":"Policy and Ethics Analyst","description":"Evaluates the legal and ethical implications of introducing defence-oriented training in K-12 settings, including alignment with human rights standards, civil liberties, and potential risks of normalizing violence. Focus areas include nonviolent defense, de-escalation, situational awareness, student welfare, parental consent, data privacy, and equitable access."},{"name":"Prof. Marcus Chen","role":"Curriculum Design and Pedagogy Analyst","description":"Develops age-appropriate, evidence-based curricula for defence-related training that emphasizes personal safety, situational awareness, de-escalation, first aid, and emergency response. Addresses pedagogy, inclusivity, accessibility, teacher training needs, assessment metrics, and integration with existing safety programs."},{"name":"Amina Yusuf","role":"Risk Governance and Community Impact Analyst","description":"Assesses societal and school-climate

In [12]:
from pprint import pprint

thread_state = await client.threads.get_state(thread["thread_id"])
print(thread_state["next"])
pprint(thread_state)

['human_feedback']
{'checkpoint': {'checkpoint_id': '1f1b035a-da81-6cfe-8001-6f135f957bd2',
                'checkpoint_ns': '',
                'thread_id': '01a09fd0-16e9-7b81-bb91-d0c179ea5880'},
 'checkpoint_id': '1f1b035a-da81-6cfe-8001-6f135f957bd2',
 'created_at': '2026-09-14T12:13:21.842677+00:00',
 'interrupts': [],
 'metadata': {'assistant_id': '275b7937-f831-5556-befa-5064e32e4bed',
              'created_by': 'system',
              'graph_id': 'research_graph',
              'langgraph_api_url': None,
              'langgraph_api_version': '0.14.0',
              'langgraph_host': 'self-hosted',
              'langgraph_plan': 'developer',
              'langgraph_version': '1.2.11',
              'parents': {},
              'run_attempt': 1,
              'run_id': '01a09fd5-da45-72b2-9661-c396d441e40b',
              'source': 'loop',
              'step': 1,
              'thread_id': '01a09fd0-16e9-7b81-bb91-d0c179ea5880',
              'user_id': 'shamoon'},
 'next':

In [14]:
await client.threads.update_state(thread["thread_id"],
                            values={"human_fb":"approve"},
                            as_node="human_feedback")

{'checkpoint': {'thread_id': '01a09fd0-16e9-7b81-bb91-d0c179ea5880',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1b0363-034c-6e65-8002-30017e8b5cbb'},
 'configurable': {'thread_id': '01a09fd0-16e9-7b81-bb91-d0c179ea5880',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1b0363-034c-6e65-8002-30017e8b5cbb'},
 'checkpoint_id': '1f1b0363-034c-6e65-8002-30017e8b5cbb'}

In [ ]:
user_input = None

async for chunk in client.runs.stream(thread["thread_id"],
                                      graph_name,
                                      input=user_input,
                                      stream_mode="messages-tuple", # this should be "values"
                                      config=config):
    if chunk.event == "values" and "final_report" in chunk.data:
        print("================ FINAL REPORT ====================")
        print(chunk.data["final_report"])

In [16]:
state = await client.threads.get_state(thread["thread_id"])
state

{'values': {'topic': 'defence training for students in schools',
  'max_num_analyst': 3,
  'human_fb': 'approve',
  'analysts': [{'name': 'Dr. Priya Nanda',
    'role': 'Policy and Ethics Analyst',
    'description': 'Evaluates the legal and ethical implications of introducing defence-oriented training in K-12 settings, including alignment with human rights standards, civil liberties, and potential risks of normalizing violence. Focus areas include nonviolent defense, de-escalation, situational awareness, student welfare, parental consent, data privacy, and equitable access.'},
   {'name': 'Prof. Marcus Chen',
    'role': 'Curriculum Design and Pedagogy Analyst',
    'description': 'Develops age-appropriate, evidence-based curricula for defence-related training that emphasizes personal safety, situational awareness, de-escalation, first aid, and emergency response. Addresses pedagogy, inclusivity, accessibility, teacher training needs, assessment metrics, and integration with existing 

In [18]:
print(state["values"]["final_report"])

Safety in schools should protect students’ dignity and rights as fiercely as their physical safety. This report asks: can defence-oriented training be implemented in a way that enhances protection without normalizing violence or eroding civil liberties? Focusing on nonviolent defense, de-escalation, and situational awareness, the analysis situates such programs within rights-based, trauma-informed, and equity-centered practices.

Grounded in human rights standards and evidence-informed design, the introduction outlines a concise framework for evaluating and guiding school-based safety training. It emphasizes consent, data governance, transparency, and independent oversight; it prescribes age-appropriate, trauma-informed curricula; and it maps governance, implementation, and evaluation to real-world school contexts. The sections that follow present the core design principles, curriculum design, governance and privacy safeguards, and a practical path from pilot to scale, with metrics and